In [1]:
import pandas as pd
import wandb

In [2]:
# # After running the setup/query cells, use this to discover logged history keys.
# history_columns = list(selected_run.history(pandas=True).columns)
# history_columns

In [3]:
KEYS_IOU_VAL = ['val/segmentation/iou/pred_vs_gt_epoch',
 'val/registration/iou/warped_vs_gt_epoch']
KEYS_VIOLATION_VAL = ['val/registration/constraint/violation_rate_epoch',
 'val/segmentation/constraint/violation_rate_epoch']
KEYS_IOU_VALEXTRA = ['val_extra/segmentation/iou/pred_vs_gt_epoch','val_extra/registration/iou/warped_vs_gt_epoch']

KEYS = KEYS_IOU_VAL + KEYS_VIOLATION_VAL + KEYS_IOU_VALEXTRA

# Weights & Biases experiment processing

> **One-time setup:** authenticate in a terminal with `wandb login`, then set `WANDB_ENTITY` and, if needed, `WANDB_PROJECT`. The code below never stores credentials in the notebook.

The first query creates a lightweight table of the most recently created runs. Use it as the starting point for filtering, comparing metrics, and downloading artifacts.

In [4]:
WANDB_ENTITY = "ksicht"
WANDB_PROJECT = "Constraints"
EXPERIMENT_TAG = "initial_decoupled"
EXPERIMENT_GROUP_PREFIX = "ex3-initial_decoupled-"

api = wandb.Api()
api.settings['project'] = WANDB_PROJECT
api.settings['entity'] = WANDB_ENTITY
project_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}"
print(f"Loading finished runs from {project_path} with tag {EXPERIMENT_TAG}")

# The tag selects this script's runs; mode, modality, and seed are encoded locally.
runs = list(
    api.runs(
        project_path,
        filters={
            "tags": {"$in": [EXPERIMENT_TAG]},
            "jobType": "train",
            "state": {"$in": ["finished"]},
        },
        order="-created_at",
    )
)
print(f"Loaded {len(runs)} finished runs from {project_path} with tag {EXPERIMENT_TAG}")

run_rows = []
for run in runs:
    group = run.group or ""
    mode, modality = group.removeprefix(EXPERIMENT_GROUP_PREFIX).rsplit("-", 1)
    seed = int(run.name.rsplit("-seed", 1)[1])
    run_summary = run.summary
    run_rows.append(
        {
            "id": run.id,
            "name": run.name,
            "group": group,
            "mode": mode,
            "modality": modality,
            "seed": seed,
            "state": run.state,
            "created_at": run.created_at,
            **{key: run_summary.get(key) for key in KEYS},
        }
    )

run_table = pd.DataFrame(run_rows)
print(f"Built a table of {len(run_table)} finished initial_decoupled runs")
run_table.sort_values(["mode", "modality", "seed"]).reset_index(drop=True)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/michal/.netrc.


Loading finished runs from ksicht/Constraints with tag initial_decoupled


KeyboardInterrupt: 

In [7]:

prd = (run_table.groupby(["mode", "modality"],dropna=False).agg(ids=("id",list)).reset_index())
prd

,mode,modality,ids
0,BCE_BCE,affine,"[dbk4dtkz, 74bzqxbl, 32pu9wpo]"
1,BCE_BCE,deformed,"[isfmyv3u, ynuwrxoy, q490jqig]"
2,BCE_BlurredLoss,affine,"[ca01dg3d, 0394fmeg, njf6gjaz]"
3,BCE_BlurredLoss,deformed,"[y485bcgn, pu4fst28, pzogbdm7]"
4,BCE_CentroidLoss,affine,"[rvyt6s2c, qngpgv3n, nk3e1tlh]"
5,BCE_CentroidLoss,deformed,"[qq1xi2qv, tf07chu0, tloszfoh]"
6,BCE_DSDF_MSE,affine,"[3en7iknk, 6jl1ar7a, lbfegmnd]"
7,BCE_DSDF_MSE,deformed,"[7tp9nryf, 6h2544xi, ksmbt558]"
8,BCE_OneSideSDFPlain,affine,"[ndcl8djc, ks9ndihf, b1f9ff0x]"
9,BCE_OneSideSDFPlain,deformed,"[urinbgyc, e7zybzcp, o4czignh]"


In [8]:
# Group by approach and show which seeds are present for each configuration.
summary = (
    run_table.groupby(["mode", "modality"], dropna=False)
    .agg(
        runs=("id", "count"),
        seeds=("seed", lambda values: sorted([v for v in values if v is not None])),
    )
    .reset_index()
)
summary.sort_values(["mode", "modality"])

,mode,modality,runs,seeds
0,BCE_BCE,affine,3,"[42, 219, 1234]"
1,BCE_BCE,deformed,3,"[42, 219, 1234]"
2,BCE_BlurredLoss,affine,3,"[42, 219, 1234]"
3,BCE_BlurredLoss,deformed,3,"[42, 219, 1234]"
4,BCE_CentroidLoss,affine,3,"[42, 219, 1234]"
5,BCE_CentroidLoss,deformed,3,"[42, 219, 1234]"
6,BCE_DSDF_MSE,affine,3,"[42, 219, 1234]"
7,BCE_DSDF_MSE,deformed,3,"[42, 219, 1234]"
8,BCE_OneSideSDFPlain,affine,3,"[42, 219, 1234]"
9,BCE_OneSideSDFPlain,deformed,3,"[42, 219, 1234]"


In [9]:
# Example: select one run for a given configuration and inspect it directly.
example_mode = "BCE_OneSideSDFSquared"
example_modality = "affine"

selected_row = (
    run_table[
        (run_table["mode"] == example_mode)
        & (run_table["modality"] == example_modality)
    ]
    .sort_values("seed")
    .iloc[0]
)

selected_run = api.run(selected_row["id"])
print("Selected run:", selected_run.name)
print("Group:", selected_run.group)
print("State:", selected_run.state)
print("Seed:", selected_run.config.get("seed"))
print("Summary keys:", list(selected_run.summary.keys()))

Selected run: ex3-initial_decoupled-BCE_OneSideSDFSquared-affine-seed42
Group: ex3-initial_decoupled-BCE_OneSideSDFSquared-affine
State: finished
Seed: 42
Summary keys: ['_step', 'epoch', '_wandb', '_runtime', 'val/epoch', '_timestamp', 'train/epoch', 'val/loss_step', 'val/loss_epoch', 'train/loss_step', 'train/loss_epoch', 'trainer/global_step', 'val_extra/loss_step', 'val_extra/loss_epoch', 'val/labels_overlay_val_s0', 'val/labels_overlay_val_s1', 'train/labels_overlay_train_s0', 'val/segmentation/iou/pred_vs_gt_step', 'val/segmentation/iou/lumen_vs_gt_step', 'val/segmentation/iou/pred_vs_gt_epoch', 'train/segmentation/iou/pred_vs_gt_step', 'val/registration/iou/warped_vs_gt_step', 'val/segmentation/iou/lumen_vs_gt_epoch', 'val/segmentation/iou/plaque_vs_gt_step', 'train/segmentation/iou/lumen_vs_gt_step', 'train/segmentation/iou/pred_vs_gt_epoch', 'val/loss/registration/one_side_sdf_step', 'val/registration/iou/warped_vs_gt_epoch', 'val/segmentation/iou/plaque_vs_gt_epoch', 'train/r

In [10]:
keys = list(selected_run.summary.keys())
s = filter(lambda k: "train/loss/registration" in k and "epoch" in k, keys)
s2 = filter(lambda k: "iou" in k and "epoch" in k and "val" in k and "val_extra"  in k, keys)
violation_keys = filter(lambda k: "violation" in k and "epoch" in k and "val" in k and "val_extra"  in k, keys)

list(s)
# list(s2)
# list(violation_keys)

['train/loss/registration/one_side_sdf_epoch']

In [14]:
# Average one logged metric across all seeds for a chosen configuration.
average_mode = "BCE_OneSideSDFSquared"
average_modality = "affine"
metric_name = "val/loss_epoch"  # Replace with a metric key from a run's summary/history.

configuration_runs = run_table[
    (run_table["mode"] == average_mode)
    & (run_table["modality"] == average_modality)
].sort_values("seed")

if configuration_runs.empty:
    raise ValueError("No finished runs match the selected mode and modality.")

histories = []
for _, run_row in configuration_runs.iterrows():
    run = api.run(f"{project_path}/{run_row['id']}")
    history = run.history(keys=["_step", metric_name], pandas=True)
    histories.append(
        history[["_step", metric_name]]
        .dropna(subset=[metric_name])
        .assign(seed=run_row["seed"])
    )

seed_histories = pd.concat(histories, ignore_index=True)
metric_by_step = (
    seed_histories.groupby("_step", as_index=False)[metric_name]
    .agg(mean="mean", std="std", runs="count")
    .rename(columns={"mean": f"{metric_name}_mean", "std": f"{metric_name}_std"})
    .sort_values("_step")
)

print(f"Averaged {len(configuration_runs)} runs for {average_mode} / {average_modality}")
metric_by_step

Averaged 3 runs for BCE_OneSideSDFSquared / affine


,_step,val/loss_epoch_mean,val/loss_epoch_std,runs
0,71,17.444343,1.191895,3
1,142,14.407729,0.680557,3
2,213,12.744478,1.127064,3
3,284,9.899578,0.698956,3
4,355,7.824034,1.348763,3
...,...,...,...,...
75,5396,0.334282,0.034342,2
76,5467,0.311649,NaN,1
77,5538,0.311301,NaN,1
78,5609,0.311086,NaN,1


In [5]:
from pathlib import Path

csv_path = Path("coupled_initial_4_8_2026.csv")
exported_runs = pd.read_csv(csv_path)
identifier_columns = {"Name", "ID", "seed", "mode", "modality"}
constraint_pairs = {
    "val/segmentation/constraint/violations": (
        "val/segmentation/constraint/violating_samples",
        "val/segmentation/constraint/total_samples",
    ),
    "val/registration/constraint/violations": (
        "val/registration/constraint/violating_samples",
        "val/registration/constraint/total_samples",
    ),
}
constraint_columns = {
    column for pair in constraint_pairs.values() for column in pair
}
iou_columns = [
    column
    for column in exported_runs.columns
    if column not in identifier_columns | constraint_columns
]
numeric_columns = iou_columns + list(constraint_columns)
exported_runs[numeric_columns] = exported_runs[numeric_columns].apply(
    pd.to_numeric, errors="coerce"
 )

def summarize_modality(modality: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    modality_runs = exported_runs[exported_runs["modality"] == modality]
    grouped = modality_runs.groupby("mode", dropna=False)
    run_counts = grouped.size()

    violation_table = pd.DataFrame({"runs": run_counts})
    for label, (violating, total) in constraint_pairs.items():
        violations = grouped[violating].sum(min_count=1)
        totals = grouped[total].sum(min_count=1)
        violation_table[label] = [
            f"{int(violations.loc[mode])} / {int(totals.loc[mode])}"
            if pd.notna(violations.loc[mode]) and pd.notna(totals.loc[mode])
            else "-"
            for mode in violation_table.index
        ]

    iou_means = grouped[iou_columns].mean()
    iou_stds = grouped[iou_columns].std()
    iou_table = pd.DataFrame({"runs": run_counts})
    for metric in iou_columns:
        iou_table[metric] = [
            (
                f"{iou_means.loc[mode, metric]:.4f} +/- "
                f"{iou_stds.loc[mode, metric]:.4f}"
                if pd.notna(iou_stds.loc[mode, metric])
                else f"{iou_means.loc[mode, metric]:.4f}"
            )
            if pd.notna(iou_means.loc[mode, metric])
            else "-"
            for mode in iou_table.index
        ]

    return (
        violation_table.reset_index().sort_values("mode").reset_index(drop=True),
        iou_table.reset_index().sort_values("mode").reset_index(drop=True),
    )

affine_violations, affine_iou = summarize_modality("affine")
deformed_violations, deformed_iou = summarize_modality("deformed")

print("Affine constraints: pooled violations / evaluated samples across seeds")
display(affine_violations)
print("Affine IoU: mean +/- sample standard deviation across seeds")
display(affine_iou)
print("Deformed constraints: pooled violations / evaluated samples across seeds")
display(deformed_violations)
print("Deformed IoU: mean +/- sample standard deviation across seeds")
display(deformed_iou)

Affine constraints: pooled violations / evaluated samples across seeds


,mode,runs,val/segmentation/constraint/violations,val/registration/constraint/violations
0,BCE_BCE,3,2 / 297,3 / 297
1,BCE_BlurredLoss,3,3 / 297,7 / 297
2,BCE_CentroidLoss,3,3 / 297,0 / 297
3,BCE_DSDF_MSE,3,3 / 297,297 / 297
4,BCE_OneSideSDFPlain,3,2 / 297,0 / 297
5,BCE_OneSideSDFSquared,3,1 / 297,2 / 297
6,BCE_SDFTEMPLATE_MSE,3,3 / 297,0 / 297
7,BCE_SDFTEMPLATE_OneSideSDFSQUARE,3,2 / 297,2 / 297
8,OneSideSDFPlain_OneSideSDFPlain,3,2 / 297,0 / 297
9,OneSideSDFSquared_OneSideSDFSquared,3,24 / 297,0 / 297


Affine IoU: mean +/- sample standard deviation across seeds


,mode,runs,val_extra/registration/iou/warped_vs_gt_epoch,val/registration/iou/warped_vs_gt_epoch,val_extra/segmentation/iou/pred_vs_gt_epoch,val/segmentation/iou/pred_vs_gt_epoch
0,BCE_BCE,3,0.6492 +/- 0.0254,0.6490 +/- 0.0256,0.9755 +/- 0.0007,0.9755 +/- 0.0007
1,BCE_BlurredLoss,3,0.6650 +/- 0.0072,0.6650 +/- 0.0071,0.9757 +/- 0.0004,0.9757 +/- 0.0004
2,BCE_CentroidLoss,3,0.7388 +/- 0.0074,0.7388 +/- 0.0075,0.9754 +/- 0.0004,0.9754 +/- 0.0004
3,BCE_DSDF_MSE,3,0.1652 +/- 0.0000,0.1652 +/- 0.0000,0.9747 +/- 0.0003,0.9747 +/- 0.0003
4,BCE_OneSideSDFPlain,3,0.7805 +/- 0.0488,0.7802 +/- 0.0487,0.9751 +/- 0.0007,0.9751 +/- 0.0007
5,BCE_OneSideSDFSquared,3,0.8032 +/- 0.0256,0.8034 +/- 0.0259,0.9757 +/- 0.0003,0.9757 +/- 0.0003
6,BCE_SDFTEMPLATE_MSE,3,0.1902 +/- 0.0039,0.1902 +/- 0.0039,0.9746 +/- 0.0002,0.9746 +/- 0.0002
7,BCE_SDFTEMPLATE_OneSideSDFSQUARE,3,0.8202 +/- 0.0231,0.8202 +/- 0.0223,0.9753 +/- 0.0006,0.9753 +/- 0.0006
8,OneSideSDFPlain_OneSideSDFPlain,3,0.8059 +/- 0.0581,0.8062 +/- 0.0562,0.9653 +/- 0.0013,0.9653 +/- 0.0013
9,OneSideSDFSquared_OneSideSDFSquared,3,0.7942 +/- 0.0105,0.7960 +/- 0.0106,0.9579 +/- 0.0024,0.9579 +/- 0.0024


Deformed constraints: pooled violations / evaluated samples across seeds


,mode,runs,val/segmentation/constraint/violations,val/registration/constraint/violations
0,BCE_BCE,3,0 / 294,177 / 294
1,BCE_BlurredLoss,3,0 / 294,129 / 294
2,BCE_CentroidLoss,3,0 / 294,4 / 294
3,BCE_DSDF_MSE,3,0 / 294,294 / 294
4,BCE_OneSideSDFPlain,3,0 / 294,13 / 294
5,BCE_OneSideSDFSquared,3,0 / 294,11 / 294
6,BCE_SDFTEMPLATE_MSE,3,0 / 294,294 / 294
7,BCE_SDFTEMPLATE_OneSideSDFSQUARE,3,0 / 294,288 / 294
8,OneSideSDFPlain_OneSideSDFPlain,3,0 / 294,6 / 294
9,OneSideSDFSquared_OneSideSDFSquared,3,0 / 294,0 / 294


Deformed IoU: mean +/- sample standard deviation across seeds


,mode,runs,val_extra/registration/iou/warped_vs_gt_epoch,val/registration/iou/warped_vs_gt_epoch,val_extra/segmentation/iou/pred_vs_gt_epoch,val/segmentation/iou/pred_vs_gt_epoch
0,BCE_BCE,3,0.3364 +/- 0.0797,0.8953 +/- 0.0585,0.9628 +/- 0.0006,0.9628 +/- 0.0006
1,BCE_BlurredLoss,3,0.3608 +/- 0.0381,0.9610 +/- 0.0012,0.9630 +/- 0.0002,0.9630 +/- 0.0002
2,BCE_CentroidLoss,3,0.4547 +/- 0.0714,0.9343 +/- 0.0036,0.9629 +/- 0.0002,0.9629 +/- 0.0002
3,BCE_DSDF_MSE,3,0.1634 +/- 0.0000,0.1634 +/- 0.0000,0.9635 +/- 0.0001,0.9635 +/- 0.0001
4,BCE_OneSideSDFPlain,3,0.3191 +/- 0.1558,0.9454 +/- 0.0049,0.9624 +/- 0.0002,0.9624 +/- 0.0002
5,BCE_OneSideSDFSquared,3,0.4472 +/- 0.0542,0.9196 +/- 0.0275,0.9627 +/- 0.0006,0.9627 +/- 0.0006
6,BCE_SDFTEMPLATE_MSE,3,0.1634 +/- 0.0000,0.1634 +/- 0.0000,0.9635 +/- 0.0001,0.9635 +/- 0.0001
7,BCE_SDFTEMPLATE_OneSideSDFSQUARE,3,0.2680 +/- 0.0397,0.8406 +/- 0.1660,0.9627 +/- 0.0007,0.9627 +/- 0.0007
8,OneSideSDFPlain_OneSideSDFPlain,3,0.4532 +/- 0.0823,0.9431 +/- 0.0048,0.9382 +/- 0.0005,0.9382 +/- 0.0005
9,OneSideSDFSquared_OneSideSDFSquared,3,0.4156 +/- 0.0602,0.9401 +/- 0.0023,0.9384 +/- 0.0026,0.9384 +/- 0.0026
